# Fraud Detection Dashboard — Matplotlib Visualizations

This notebook generates all dashboard-level visualizations for the
Real-Time Big Data Analytics Fraud Detection project.

**Sections:**
1. KPI Summary Metrics
2. Time-Series Analysis (Transaction volume & fraud over time)
3. Fraud Breakdown by Region, Channel, Merchant
4. Risk Score Analysis
5. Model Performance (ROC, Confusion Matrix, Feature Importance)
6. PCA Anomaly Clusters

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from sklearn.metrics import confusion_matrix, roc_curve, auc
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.append("..")

from backend.pipeline.data_ingestion import TransactionProducer
from backend.pipeline.feature_engineering import FeatureEngineer
from backend.models.isolation_forest import IsolationForestDetector
from backend.models.xgboost_model import XGBoostFraudDetector
from backend.models.ensemble import FraudEnsemble

sns.set_theme(style="darkgrid")
plt.rcParams.update({
    "figure.facecolor": "#0d1117",
    "axes.facecolor": "#161b22",
    "text.color": "#e6edf3",
    "axes.labelcolor": "#8b949e",
    "xtick.color": "#6e7681",
    "ytick.color": "#6e7681",
    "grid.color": "#21262d",
    "figure.figsize": (14, 6)
})

## 1. Generate Dataset

In [ ]:
print("Generating 10000 transactions...")
producer = TransactionProducer(seed=42)
txns = [producer.generate() for _ in range(10000)]
df = pd.DataFrame(txns)
df["timestamp"] = pd.to_datetime(df["timestamp"])
print(f"Total: {len(df)} | Fraud: {df['is_fraud'].sum()} ({df['is_fraud'].mean()*100:.2f}%)")

## 2. KPI Summary

In [ ]:
total = len(df)
fraud_count = df['is_fraud'].sum()
fraud_rate = df['is_fraud'].mean() * 100
avg_risk = df['risk_score'].mean()
avg_amount = df['amount'].mean()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
kpis = [
    (f"Total Transactions", f"{total:,}", "#4cc9f0"),
    (f"Fraud Detected", f"{fraud_count:,}", "#f72585"),
    (f"Fraud Rate", f"{fraud_rate:.2f}%", "#f72585"),
    (f"Avg Risk Score", f"{avg_risk:.1f}", "#00f5d4"),
]
for ax, (label, value, color) in zip(axes, kpis):
    ax.text(0.5, 0.6, value, fontsize=36, fontweight="bold",
            ha="center", va="center", color=color, transform=ax.transAxes)
    ax.text(0.5, 0.25, label, fontsize=12, ha="center",
            va="center", color="#8b949e", transform=ax.transAxes)
    ax.set_facecolor("#0d1117")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_color("#21262d")
plt.suptitle("KPI Dashboard — Fraud Detection Pipeline",
             fontsize=16, color="#e6edf3", y=1.05)
plt.tight_layout()
plt.savefig("../kpi_dashboard.png", dpi=150, bbox_inches="tight",
            facecolor="#0d1117")
plt.show()

## 3. Time-Series: Transactions Over 24 Hours

In [ ]:
df["hour"] = df["timestamp"].dt.hour
hourly_stats = df.groupby("hour").agg(
    total=("transaction_id", "count"),
    fraud=("is_fraud", "sum")
).assign(fraud_rate=lambda x: x["fraud"] / x["total"] * 100)

fig, ax1 = plt.subplots(figsize=(16, 6))
ax1.bar(hourly_stats.index, hourly_stats["total"], color="#4cc9f0",
        alpha=0.6, label="Total Transactions", width=0.8)
ax1.set_xlabel("Hour of Day", fontsize=12)
ax1.set_ylabel("Transaction Count", color="#4cc9f0", fontsize=12)
ax1.tick_params(axis="y", labelcolor="#4cc9f0")

ax2 = ax1.twinx()
ax2.plot(hourly_stats.index, hourly_stats["fraud"], color="#f72585",
         marker="o", linewidth=2.5, label="Fraud Count", markersize=6)
ax2.fill_between(hourly_stats.index, hourly_stats["fraud"], alpha=0.15,
                 color="#f72585")
ax2.set_ylabel("Fraud Count", color="#f72585", fontsize=12)
ax2.tick_params(axis="y", labelcolor="#f72585")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
plt.title("24-Hour Transaction Volume & Fraud Count", fontsize=14, color="#e6edf3")
plt.tight_layout()
plt.savefig("../timeseries_transactions.png", dpi=150, bbox_inches="tight",
            facecolor="#0d1117")
plt.show()

## 4. Fraud Rate by Region and Channel

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

region_rate = df.groupby("region")["is_fraud"].mean().sort_values() * 100
colors_reg = ["#f72585" if v > 10 else "#4cc9f0" for v in region_rate.values]
axes[0].barh(region_rate.index, region_rate.values, color=colors_reg, alpha=0.8)
axes[0].set_title("Fraud Rate by Region", fontsize=14, color="#e6edf3")
axes[0].set_xlabel("Fraud Rate (%)")
for i, v in enumerate(region_rate.values):
    axes[0].text(v + 0.3, i, f"{v:.1f}%", va="center", fontsize=9, color="#e6edf3")

channel_rate = df.groupby("channel")["is_fraud"].mean().sort_values() * 100
colors_ch = ["#f72585" if v > 10 else "#4cc9f0" for v in channel_rate.values]
axes[1].barh(channel_rate.index, channel_rate.values, color=colors_ch, alpha=0.8)
axes[1].set_title("Fraud Rate by Channel", fontsize=14, color="#e6edf3")
axes[1].set_xlabel("Fraud Rate (%)")
for i, v in enumerate(channel_rate.values):
    axes[1].text(v + 0.3, i, f"{v:.1f}%", va="center", fontsize=9, color="#e6edf3")

plt.tight_layout()
plt.savefig("../fraud_rate_region_channel.png", dpi=150,
            bbox_inches="tight", facecolor="#0d1117")
plt.show()

## 5. Fraud Heatmap: Region × Channel

In [ ]:
heatmap_data = df.pivot_table(
    index="region", columns="channel", values="is_fraud", aggfunc="mean"
) * 100

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(heatmap_data, annot=True, fmt=".1f", cmap="Reds",
            ax=ax, cbar_kws={"label": "Fraud Rate (%)"},
            linewidths=0.5, linecolor="#21262d")
ax.set_title("Fraud Rate Heatmap — Region × Channel", fontsize=14, color="#e6edf3")
ax.set_xlabel("Channel")
ax.set_ylabel("Region")
plt.tight_layout()
plt.savefig("../fraud_heatmap.png", dpi=150, bbox_inches="tight",
            facecolor="#0d1117")
plt.show()

## 6. Merchant Risk Analysis

In [ ]:
merchant_stats = df.groupby("merchant").agg(
    total_txns=("transaction_id", "count"),
    fraud_count=("is_fraud", "sum"),
    avg_amount=("amount", "mean")
).assign(fraud_rate=lambda x: x["fraud_count"] / x["total_txns"] * 100)
merchant_stats = merchant_stats.sort_values("fraud_rate", ascending=True)

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(merchant_stats.index, merchant_stats["fraud_rate"],
               color="#f72585", alpha=0.7)
ax.set_title("Fraud Rate by Merchant", fontsize=14, color="#e6edf3")
ax.set_xlabel("Fraud Rate (%)")
for bar, rate in zip(bars, merchant_stats["fraud_rate"]):
    ax.text(rate + 0.2, bar.get_y() + bar.get_height()/2,
            f"{rate:.1f}%", va="center", fontsize=9, color="#e6edf3")
plt.tight_layout()
plt.savefig("../merchant_risk.png", dpi=150, bbox_inches="tight",
            facecolor="#0d1117")
plt.show()

## 7. Model Training & Evaluation

In [ ]:
print("Training ML models...")
engineer = FeatureEngineer()
X = engineer.batch_transform(txns)
y = df["is_fraud"].values.astype(int)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

# Train models
if_model = IsolationForestDetector(contamination=df["is_fraud"].mean())
if_model.train(X_train.values)
if_preds = if_model.predict(X_test.values)
if_probs = if_model.predict_proba(X_test.values)

xgb_model = XGBoostFraudDetector(
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum()
)
xgb_model.train(X_train.values, y_train, X_test.values, y_test)
xgb_preds = xgb_model.predict(X_test.values)
xgb_probs = xgb_model.predict_proba(X_test.values)

# ROC Curves
if_fpr, if_tpr, _ = roc_curve(y_test, if_probs)
xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xgb_probs)
if_auc = auc(if_fpr, if_tpr)
xgb_auc = auc(xgb_fpr, xgb_tpr)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ROC
axes[0].plot(if_fpr, if_tpr, label=f"Isolation Forest (AUC={if_auc:.3f})",
             color="#4cc9f0", lw=2)
axes[0].plot(xgb_fpr, xgb_tpr, label=f"XGBoost (AUC={xgb_auc:.3f})",
             color="#f72585", lw=2)
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3, color="#6e7681")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curves — Model Comparison", fontsize=14, color="#e6edf3")
axes[0].legend()

# Confusion Matrices
cm_xgb = confusion_matrix(y_test, xgb_preds)
sns.heatmap(cm_xgb, annot=True, fmt="d", cmap="viridis", ax=axes[1],
            xticklabels=["Clean", "Fraud"], yticklabels=["Clean", "Fraud"])
axes[1].set_title("XGBoost Confusion Matrix", fontsize=14, color="#e6edf3")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.savefig("../model_performance.png", dpi=150, bbox_inches="tight",
            facecolor="#0d1117")
plt.show()

## 8. Feature Importance (XGBoost)

In [ ]:
importance = xgb_model.get_feature_importance()
fig, ax = plt.subplots(figsize=(14, 6))
feats = list(importance.keys())
vals = list(importance.values())
colors_feat = ["#f72585" if v > 0.08 else "#4cc9f0" for v in vals]
ax.barh(feats, vals, color=colors_feat, alpha=0.8)
ax.set_title("XGBoost Feature Importance", fontsize=14, color="#e6edf3")
ax.set_xlabel("Importance Score")
for i, v in enumerate(vals):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9, color="#e6edf3")
plt.tight_layout()
plt.savefig("../feature_importance.png", dpi=150, bbox_inches="tight",
            facecolor="#0d1117")
plt.show()

## 9. Anomaly Clusters (PCA Projection)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(14, 8))
ax.scatter(X_pca[~y.astype(bool), 0], X_pca[~y.astype(bool), 1],
           c="#4cc9f0", alpha=0.25, s=8, label="Clean Transactions")
ax.scatter(X_pca[y.astype(bool), 0], X_pca[y.astype(bool), 1],
           c="#f72585", alpha=0.6, s=20, label="Fraud Transactions",
           edgecolors="white", linewidth=0.3)
ax.set_title("Anomaly Clusters — PCA Projection of Engineered Features",
             fontsize=14, color="#e6edf3")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
ax.legend(markerscale=2)
plt.tight_layout()
plt.savefig("../anomaly_clusters.png", dpi=150, bbox_inches="tight",
            facecolor="#0d1117")
plt.show()

## 10. Fraud Type Distribution

In [ ]:
fraud_data = df[df["is_fraud"]]
fraud_type_counts = fraud_data["fraud_type"].value_counts()

fig, ax = plt.subplots(figsize=(12, 6))
colors_types = ["#f72585", "#4cc9f0", "#00f5d4", "#e6a919", "#8b5cf6", "#ff6b6b"]
wedges, texts, autotexts = ax.pie(
    fraud_type_counts.values, labels=fraud_type_counts.index,
    autopct="%1.1f%%", startangle=90, colors=colors_types[:len(fraud_type_counts)],
    textprops={"color": "#e6edf3", "fontsize": 11}
)
ax.set_title("Fraud Type Distribution", fontsize=14, color="#e6edf3")
plt.tight_layout()
plt.savefig("../fraud_type_distribution.png", dpi=150, bbox_inches="tight",
            facecolor="#0d1117")
plt.show()

## Summary
This notebook generated visualizations for all key aspects of the fraud detection pipeline:
- KPI metrics and 24-hour transaction trends
- Fraud patterns by region, channel, and merchant
- ML model performance (ROC, confusion matrix)
- Feature importance and anomaly clusters
- Fraud type breakdown

All charts are saved as PNG images in the project root directory.